> MapReduce 面试速成指南 — DE (NG / Senior) 2026


## 1. Core Concept & Programming Model

MapReduce 是 Google 在 2004 年提出的分布式计算编程模型，核心思想极其简单：

- **Map Phase**：把输入数据拆成独立的 chunk，每个 chunk 被一个 mapper 处理，输出 `(key, value)` pairs
- **Reduce Phase**：相同 key 的所有 values 被聚合到同一个 reducer，执行最终计算

```text
Input → Split → Map → Shuffle & Sort → Reduce → Output
```

**一句话本质**：MapReduce 是一种 **"分而治之 + 按 key 聚合"** 的抽象。你在面试中说的任何分布式数据处理逻辑，本质上都可以用这个框架去拆解。

---



## 2. End-to-End Job Lifecycle（完整执行流程）

<img src='./pic/4_mapreduce_lifecycle.png' width=500>

面试中被问到 "walk me through a MapReduce job" 时，按以下顺序讲：

### 2.1 Input Splitting

- **InputFormat** 决定如何切分数据。默认 `TextInputFormat` 按 HDFS block（128MB）切分
- 每个 **input split** 对应一个 **map task**
- split 是逻辑概念，不是物理拷贝；它记录的是 offset + length

💡 面试加分点：  
- split 和 block 不完全对齐时，会有跨节点读取（remote read），
- 但通常 HDFS 会尽量保证 **data locality** -> 把 map task 调度到数据所在的节点上。

### 2.2 Map Phase

```text
map(key, value) → list of (intermediate_key, intermediate_value)
```

- 每个 mapper 独立处理一个 split，输出中间 KV pairs
- 输出先写到 **内存 buffer**（默认 100MB，`io.sort.mb`），到 80% 阈值时 **spill to disk**
- 每次 spill 前会做：**partition → sort → (optional) combine**
- 多次 spill 的文件最终 **merge** 成一个排好序的文件

### 2.3 Shuffle & Sort（最关键的阶段）

这是 MapReduce 性能瓶颈的核心所在：

**Map 端（Shuffle Write）**：
1. 每个 KV pair 经过 **Partitioner** 决定去哪个 reducer（默认 `HashPartitioner`：`key.hashCode() % numReducers`）
2. 在每个 partition 内部 **sort** by key
3. 如果设了 **Combiner**，在排序后做局部聚合（mini-reduce）

**Reduce 端（Shuffle Read）**：
1. 每个 reducer 通过 HTTP 从所有 mapper 拉取属于自己 partition 的数据（**copy phase**）
2. 拉过来的数据做 **merge-sort**（先内存 merge，内存不够再 spill 到磁盘）
3. merge 完成后输入 reduce 函数

🔥 高频面试题："**Why is shuffle expensive?**"     
回答：因为它涉及 **磁盘 I/O**（map 端 spill + reduce 端 merge）、**网络传输**（所有 mapper 到所有 reducer 的数据搬运）、以及 **排序**。这三者叠加构成了 MapReduce 最大的开销。

### 2.4 Reduce Phase

```text
reduce(key, list_of_values) → list of (output_key, output_value)
```

- 输入已经按 key 排好序，相同 key 的 values 连续排列
- 输出写到 HDFS（每个 reducer 写一个 part 文件）

### 2.5 Output

- 默认 `TextOutputFormat`，输出到 HDFS
- 每个 reducer 产生一个 `part-r-00000`, `part-r-00001`, ... 文件
- 如果下游需要一个整文件，需要额外做 merge（或者只用一个 reducer，但会丧失并行性）

---



## 3. Key Components Deep Dive

### 3.1 Combiner（面试必知）

- 本质是 **map 端的 mini-reducer**，在数据离开 mapper 之前做局部聚合
- **目的**：减少 shuffle 阶段的网络传输量
- **限制**：只能用于 **commutative + associative** 操作（如 sum, count, max），不能用于 mean（因为局部平均值再平均 ≠ 全局平均值）
- 面试经典问题："Can you use a combiner for computing average?" → 不能直接用，但可以改为输出 `(sum, count)` 再在 reducer 里算 `sum/count`

### 3.2 Partitioner

- 决定每个 KV pair 发到哪个 reducer
- 默认 `HashPartitioner`：`(key.hashCode() & Integer.MAX_VALUE) % numReduceTasks`
- **Custom Partitioner** 的场景：
  - 解决 data skew（见第 5 节）
  - 实现 **total order**（如 TotalOrderPartitioner，配合 sampling）
  - 业务需求：比如按 region 分 partition

### 3.3 InputFormat / OutputFormat

| 类型 | 说明 |
|------|------|
| `TextInputFormat` | 按行读取，key 是 byte offset，value 是行内容 |
| `KeyValueTextInputFormat` | 按分隔符拆 key 和 value |
| `SequenceFileInputFormat` | 读 Hadoop 二进制格式，支持压缩 |
| `NLineInputFormat` | 每 N 行作为一个 split |.   



> 面试中一般不会深挖 InputFormat，但知道默认行为和 SequenceFile 是加分项。

### 3.4 Speculative Execution

- 当某个 task 运行明显慢于同阶段其他 task 时，框架会在另一个节点启动 **备份 task（backup task）**
- 谁先完成用谁的结果，另一个被 kill
- 解决的问题：**straggler**（因硬件故障、资源争用等导致的慢 task）
- 注意：对于有 **side effect** 的 task（如写外部数据库），speculative execution 可能导致重复写入

### 3.5 Job vs task vs tasktracker

这三个概念是 MapReduce 执行层级的核心，面试中经常被混淆或追问，按层级从高到低理清：  
1. **Job** 是用户提交的一个**完整的 MapReduce 程序**。
   - 比如你跑一个 WordCount，从输入到输出，整个就是一个 job。
   - 一个 job 包含<u>所有的 map tasks + 所有的 reduce tasks + 配置信息 + input/output 路径</u>。
   - Job 由 JobClient 提交，分配一个**唯一的 Job ID**（如 job_202603_0001）。

2. **Task** 是 job 被拆分后的**最小执行单元**。
   - 一个 job 会被拆成 **N 个 map tasks + M 个 reduce tasks**。
     - 每个 map task 处理*一个 input split*，
     - 每个 reduce task 处理*一组 key partition*。
   - Task 是容错的基本单位
     - 某个 task 失败了，框架重试的是这个 task，不是整个 job。
   - 还有一种特殊的 task 叫 **task attempt**，就是同一个 task 的多次尝试（失败重试或 speculative execution 都会产生新的 attempt）。

3. **TaskTracker** 是 MRv1 架构中每个 worker 节点上的守护进程，负责执行 **JobTracker** 分配过来的 tasks。
   - 它定期向 JobTracker 发 heartbeat 汇报自身状态（有多少 slot 可用、task 运行情况等）。
   - 每个 TaskTracker 有固定数量的 map slots 和 reduce slots，task 就运行在这些 slot 里。

它们的层级关系：
1. **MRv1 架构**中是 
   - JobTracker → TaskTracker → Task 的三层结构。
   - JobTracker 是 master，负责 job 调度和 task 分配；
   - TaskTracker 是 worker，负责执行 task 并汇报状态。
   - 这个架构的核心问题是 JobTracker 既管资源又管调度，是 single point of failure，而且 slot 是静态划分的（map slot 和 reduce slot 不能互借），资源利用率低。
2. **MRv2 (YARN) 架构**中，
   - TaskTracker 被淘汰了，取而代之的是 **NodeManager + Container** 的模型。
   - 对应关系是：JobTracker 的资源管理职责 → ResourceManager，JobTracker 的 job 管理职责 → 每个 job 自己的 ApplicationMaster，TaskTracker → NodeManager，固定 slot → 动态 Container（CPU + Memory 灵活分配）。

面试中如果被问到这个话题，关键是说清两点：一是 MRv1 的 JobTracker 是瓶颈和 SPOF，二是 YARN 通过职责分离解决了这个问题，并且让集群可以跑 Spark、Flink 等多种计算框架，不再只服务 MapReduce。

---



## 4. Fault Tolerance（容错机制）

### Task-Level Fault Tolerance

- 如果一个 map/reduce task 失败，**ApplicationMaster**（YARN 架构下）会在另一个节点重新调度该 task
- 默认重试次数：`mapreduce.map.maxattempts = 4`
- Map 的输出存在 local disk，如果 mapper 所在节点挂了，已完成的 map task 也需要 **重新执行**（因为 reduce 还没拉完数据）
- Reduce 的输出存在 HDFS（有 replication），所以 reduce 完成后不需要重跑

### 与 Spark 容错的对比

| 维度 | MapReduce | Spark |
|------|-----------|-------|
| 容错单位 | Task 重试 | RDD lineage 重算 |
| 中间数据 | 落盘到 local disk | 默认在内存，可选 persist 到磁盘 |
| 恢复代价 | 重跑单个 task | 重算丢失 partition 的 lineage chain |
| 优势场景 | 超大数据 + 长 pipeline | 迭代计算 + 多阶段 DAG |

面试话术：
- MapReduce's disk-based fault tolerance is more expensive per-operation but more robust for very large, one-pass batch jobs. 
- Spark's lineage-based recovery is cheaper for iterative workloads but can cause cascading recomputation if the lineage is deep.

---



## 5. Data Skew（数据倾斜）— 高频考点

### 什么是 Data Skew？

某些 key 的数据量远大于其他 key，导致对应的 reducer 处理时间远超其他 reducer，整个 job 的完成时间取决于最慢的那个 reducer。

### 如何识别？

- Job counter 显示某个 reduce task 处理的记录数远高于其他
- Reduce 阶段有一两个 task "卡住"很久，其他早已完成
- 在 **Spark UI** 或 **YARN UI** 上看到 task duration 分布极不均匀

### 解决方案（面试重点）

#### 方案 1：Salting Key

```text
原始 key: "hot_key"
加盐后: "hot_key_0", "hot_key_1", ..., "hot_key_9"
```

- 第一轮 MR：key 加随机后缀 → 分散到多个 reducer 做局部聚合local aggregation
- 第二轮 MR：去掉后缀 → 全局聚合global aggregation
- **本质**：用两轮 job 换取更均匀的数据分布

#### 方案 2：Custom Partitioner

- 对已知的 hot key 做特殊路由
- 比如把 `null` key 或已知高频 key 随机分配到多个 reducer

#### 方案 3：Map-Side Aggregation（Combiner）

- 如果操作支持 combiner，在 map 端先聚合，能大幅减少 hot key 进入 shuffle 的数据量

#### 方案 4：Increase Parallelism

- 增加 reducer 数量（`mapreduce.job.reduces`），让 hash 分布更均匀
- 不能根本解决问题，但在 skew 不严重时有效

#### 方案 5：Separate Processing

- 把 hot key 单独拿出来处理（比如 broadcast 到所有 mapper 做 map-side join），剩下的走正常 reduce

面试模板回答：   
When I encounter data skew,   
- my first step is to identify the hot keys via job counters or profiling. 
- Then depending on the operation, 
  - I'd consider salting the key for a two-pass aggregation, 
  - using a combiner for map-side pre-aggregation, 
  - or switching to a map-side join if one side is small enough to broadcast.

---



## 6. Join Patterns（必考）

### 6.1 Reduce-Side Join

```text
Map: 两张表的 mapper 分别输出 (join_key, tagged_value)
     tag 用来标记数据来自哪张表
Shuffle: 相同 join_key 发到同一个 reducer
Reduce: 在 reducer 中做 cross product / merge
```

- **优点**：通用，不限制数据大小
- **缺点**：所有数据都要走 shuffle，网络开销大
- **适用**：两张大表 join

### 6.2 Map-Side Join（Broadcast Join）

```text
Setup: 把小表加载到每个 mapper 的内存（通过 DistributedCache）
Map: 对大表的每条记录，在内存里查小表做 join
```

- **优点**：完全避免 shuffle，速度极快
- **缺点**：小表必须能放进内存
- **适用**：一大一小表 join（小表通常 < 几百 MB）

面试经典问题："**When would you choose map-side join over reduce-side join?**"      
回答：   
- When one of the datasets is small enough to fit in memory and can be broadcasted to all mappers via DistributedCache. 
- This eliminates the shuffle entirely, which is the most expensive part of a MapReduce job.

### 6.3 Semi-Join

- 先从小表提取 join key 集合，broadcast 到 mapper
- Mapper 过滤大表，只保留 key 在集合中的记录
- 再做 reduce-side join（数据量已大幅减少）
- **适用**：两张表都很大，但 join 后结果集很小（高 selectivity）

### 6.4 Secondary Sort

- 需求：在 reduce 中，相同 key 的 values 按某个字段排序
- 实现：构造 **composite key**（如 `(natural_key, sort_key)`），自定义 Partitioner 只按 `natural_key` 分区，自定义 **SortComparator** 按完整 composite key 排序，自定义 **GroupingComparator** 按 `natural_key` 分组
- **用途**：时序数据处理、sessionization

---



## 7. MapReduce vs Spark（必问对比）

### 核心架构差异

| 维度 | MapReduce | Spark |
|------|-----------|-------|
| 执行模型 | 固定两阶段 (Map → Reduce) | DAG 任意多阶段 |
| 中间数据 | 每个 stage 落盘 (HDFS / local disk) | 内存优先，可选落盘 |
| 延迟 | 高（磁盘 I/O 重） | 低（内存计算） |
| 迭代计算 | 差（每次迭代都要读写磁盘） | 强（cache RDD 在内存中） |
| 编程模型 | 低级 API (map/reduce) | 高级 API (DataFrame, SQL, ML) |
| 资源管理 | 依赖 YARN | 自带 standalone + 支持 YARN/K8s/Mesos |
| Shuffle | 必须落盘 | 也落盘（但 sort-based shuffle 更优化） |

### 什么时候 MapReduce 反而更合适？

1. **超大规模单次批处理**：数据量大到 Spark executor 内存装不下，MapReduce 的落盘模式反而更稳定
2. **集群资源有限**：MapReduce 对内存要求低
3. **遗留系统**：Hive on MR、Pig 等老系统仍在运行
4. **极端容错要求**：每个 stage 落盘意味着失败时恢复代价小

面试话术：  
- MapReduce trades latency for stability 
  - every intermediate result is materialized to disk, which means higher I/O cost per job but lower recovery cost on failure. 
- Spark inverts this tradeoff by keeping data in memory, 
  - which is ideal for iterative workloads like ML training 
  - but can struggle under memory pressure with very large shuffles.

---



## 8. YARN Architecture（加分项）

> details in /overview/.._Hadoop..

MapReduce 2.0 运行在 YARN 之上：

```text
ResourceManager (RM)
├── Scheduler：分配资源
└── ApplicationsManager：管理 application 提交

NodeManager (NM)：每个节点一个，管理 container

ApplicationMaster (AM)：每个 job 一个
├── 向 RM 申请资源
├── 分配 map/reduce task 到 container
└── 监控 task 状态，处理失败重试
```

**面试要点**：
- 与 MRv1 的区别：MRv1 的 JobTracker 既管资源又管调度，是 single point of failure；YARN 把这两个职责分开了
- YARN 不只跑 MapReduce，还能跑 Spark、Flink、Tez 等
- Container 是 YARN 的资源抽象单位（CPU + Memory）

---



## 9. Performance Tuning（调优知识点）

### Map 端调优

| 参数 | 说明 |
|------|------|
| `io.sort.mb` | Map 输出缓冲区大小（默认 100MB），增大可减少 spill 次数 |
| `io.sort.spill.percent` | 缓冲区使用到多少比例触发 spill（默认 0.8） |
| `mapreduce.map.output.compress` | 开启 map 输出压缩（推荐 Snappy/LZ4），减少 shuffle 数据量 |
| `mapreduce.input.fileinputformat.split.maxsize` | 控制 split 大小，间接控制 mapper 数量 |

### Reduce 端调优

| 参数 | 说明 |
|------|------|
| `mapreduce.reduce.shuffle.input.buffer.percent` | reduce 端用于 shuffle 数据的 heap 比例 |
| `mapreduce.reduce.shuffle.parallelcopies` | 并行 copy 线程数（默认 5），增大可加速 shuffle |
| `mapreduce.job.reduces` | reducer 数量，太少会导致 skew，太多会产生过多小文件 |

### 通用原则

- **减少 shuffle 数据量**：用 Combiner、开压缩、过滤不需要的字段
- **避免小文件**：大量小文件 = 大量 mapper = 大量 task 启动开销。用 `CombineFileInputFormat` 合并小文件
- **合理设置 reducer 数量**：经验法则是每个 reducer 处理 1-2 GB 数据
- **开启 JVM reuse**：`mapreduce.job.jvm.numtasks = -1`，避免反复启动 JVM 的开销

---



## 10. Classic MapReduce Patterns（编程模式）

### 10.1 Word Count（基础中的基础）

```text
Map:  (line_offset, "hello world hello") → ("hello", 1), ("world", 1), ("hello", 1)
Reduce: ("hello", [1, 1]) → ("hello", 2)
        ("world", [1]) → ("world", 1)
```

### 10.2 Inverted Index

反转索引

```text
Map:  (doc_id, "cat sat on mat") → ("cat", doc1), ("sat", doc1), ...
Reduce: ("cat", [doc1, doc3, doc7]) → ("cat", "doc1,doc3,doc7")
```

### 10.3 Top-N

```text
Map: 每个 mapper 维护一个 local top-N (用 min-heap)
     cleanup() 阶段输出 local top-N
Reduce: 单个 reducer 合并所有 mapper 的 local top-N → global top-N
```

关键：如果直接把所有数据发到一个 reducer 做 top-N，会严重瓶颈。必须在 map 端先做局部 top-N。

### 10.4 Distinct Count

```text
Map:  (line, "A B A C B") → ("A", null), ("B", null), ("A", null), ("C", null), ("B", null)
Reduce: 每个 key 只出现一次（因为 reduce 按 key 聚合）→ count keys
```

近似去重可以用 **HyperLogLog**（面试加分）。

### 10.5 Moving Average / Sessionization

- 需要 **Secondary Sort**：按 user_id partition，按 timestamp 排序
- Reducer 中按时间顺序遍历 events，维护窗口 / 检测 session gap

---



## 11. MapReduce in System Design（系统设计中的应用）

面试中被要求设计大规模数据处理系统时，用 MapReduce 思维拆解：

### 例题：Design a log analytics pipeline

```text
Stage 1 (Map): Parse raw logs → extract (timestamp, url, user_id, status_code, latency)
               Filter out health checks and bot traffic
Stage 1 (Reduce): Aggregate by url → (url, count, avg_latency, error_rate)

Stage 2 (Map): Read aggregated data → emit/i'mɪt/ (time_bucket, url, metrics)
Stage 2 (Reduce): Per time_bucket aggregation → time series data

Output → Write to columnar store (Parquet on S3) for downstream BI queries
```

### 例题：Design a large-scale URL deduplication system

```text
Map: (url) → (normalized_url, metadata)
     做 URL normalization（去 trailing slash、统一大小写、去 tracking params）
Reduce: 每个 normalized_url 只保留一条（或 merge metadata）
```

### 例题：Build a recommendation system (offline feature computation)

```text
Map: (user_id, event_log) → (user_id, feature_vector_partial)
     提取用户行为特征（点击次数、观看时长、类别偏好）
Combine: Local aggregate feature vectors
Reduce: (user_id, [all partials]) → (user_id, complete_feature_vector)
Output → Feature Store (用于在线 serving)
```

🎯 面试技巧：即使最终方案用 Spark/Flink 实现，先用 MapReduce 思维拆解问题 → 再说明 "in practice I'd use Spark for the in-memory advantage" → 展示你理解底层原理。

---



## 12. Quick-Fire Q&A（高频面试问答）

### Q: What happens if a mapper is slower than others?
**A:** 
- Speculative execution kicks in 
  - the framework launches a backup task on another node. 
- Whichever finishes first wins, the other gets killed. 
- This addresses **stragglers** caused by hardware degradation or resource contention.

### Q: Can the number of mappers be directly configured?
**A:** 
- 不能直接设置。
- Mapper 数量由 input splits 决定（= input size / split size）。
- 你可以间接控制：调整 split size、使用 `CombineFileInputFormat` 合并小文件。

### Q: What if a reducer gets OOM Out Of Memory?
**A:** 
- 通常说明 data skew 或 reducer 数量太少。
- 解决方案：增加 reducer 数、salting hot keys、增大 reduce task 的内存配置（`mapreduce.reduce.memory.mb`）。

### Q: Difference between Sort and Shuffle?
**A:** 
- Shuffle 是数据从 mapper 搬运到 reducer 的整个过程（包括 partition, sort, copy, merge）。
- Sort 是 shuffle 中的一个步骤
  - 在 map 端对输出排序，在 reduce 端对拉来的数据做 merge-sort。

### Q: Why does MapReduce sort the data?
**A:** 排序使得相同 key 的 records 物理上连续，这样 reducer 可以用 **streaming** 方式处理（不需要把一个 key 的所有 values 放进内存），内存效率极高。

### Q: How does MapReduce handle node failure?
**A:** 
- **Map node fails**: ApplicationMaster 在其他节点重跑该 map task（即使已完成的也要重跑，因为 map 输出在 local disk，节点挂了就丢了）
- **Reduce node fails**: 重跑该 reduce task，重新从 mapper 拉数据
- **ApplicationMaster fails**: ResourceManager 重启一个新的 AM

### Q: When would you NOT use MapReduce?
**A:**
- **低延迟查询**：用 Presto/Trino/Impala ➡️ 这三个本质上都属于 👉 SQL Query Engine（查询引擎）
- **实时流处理**：用 Flink/Kafka Streams
- **迭代算法（ML）**：用 Spark MLlib
- **图计算**：用 Pregel/GraphX
- **小数据**：单机就能搞定的不需要分布式开销

---



## 13. Terminology Cheat Sheet

| English Term | 中文 | 一句话解释 |
|---|---|---|
| Input Split | 输入分片 | 逻辑切分，每个 split 对应一个 map task |
| Mapper | 映射器 | 处理一个 split，输出 KV pairs |
| Reducer | 归约器 | 接收相同 key 的所有 values，做聚合 |
| Combiner | 合并器 | Map 端的局部 reduce，减少 shuffle 数据量 |
| Partitioner | 分区器 | 决定 KV pair 发到哪个 reducer |
| Shuffle | 洗牌 | 数据从 mapper 到 reducer 的整个搬运过程 |
| Spill | 溢写 | Map 缓冲区满后写到磁盘 |
| Speculative Execution | 推测执行 | 对 straggler 启动备份 task |
| Data Locality | 数据本地性 | 把计算调度到数据所在节点 |
| Straggler | 掉队者 | 运行特别慢的 task |
| Data Skew | 数据倾斜 | Key 分布不均导致负载不均 |
| Salting | 加盐 | 给 key 加随机后缀打散分布 |
| DistributedCache | 分布式缓存 | 把小文件广播到所有 task 节点 |
| Secondary Sort | 二次排序 | 在 reduce 中对 values 也排序 |
| Composite Key | 复合键 | 由多个字段组合的 key |
| Side Effect | 副作用 | Task 对外部系统的写操作 |
| Materialization | 物化 | 把中间结果写到磁盘 |
| Lineage | 血统/谱系 | Spark 中 RDD 的转换链（对比用） |
| Barrier | 屏障 | Map 全部完成后才能开始 reduce 的同步点 |

---



## 14. 面试答题框架模板

当被问到 MapReduce 相关的开放式问题时，用这个框架组织回答：

```text
1. State the core problem（明确问题）
   "The challenge here is..."

2. MapReduce decomposition（拆解成 Map 和 Reduce）
   "In the map phase, we'd... In the reduce phase, we'd..."

3. Identify bottlenecks（识别瓶颈）
   "The main bottleneck would be... because..."

4. Optimization（优化方案）
   "To address this, we could use a combiner / custom partitioner / map-side join..."

5. Modern alternative（现代替代方案）
   "In practice, I'd implement this in Spark/Flink for... but the underlying logic remains the same."
```